In [103]:
from __future__ import annotations

In [104]:
import json
import os.path
from collections.abc import Callable
from dataclasses import dataclass, field
from enum import Enum, auto
from statistics import mean
from typing import Literal, NamedTuple

import numpy as np
import pandas as pd
from jaxtyping import Float
from scipy.stats import t

from tiu_phi_3_5_mini.data_management import load_train_validation_splits, ActivationsDataSelector, \
    train_scenarios_spec_path, ProbeTrainScenario
from tiu_phi_3_5_mini.direction_learning import ReconLosses
from tiu_phi_3_5_mini.evaluation_utils import MetricsForDatasetProbes
from tiu_phi_3_5_mini.logging_setup import create_logger
from tiu_phi_3_5_mini.phi_3_5_constants import dsets_index_path, \
    directions_reconstruction_losses_path, separation_by_layer_analysis_path, train_split_classification_metrics_path, \
    validation_split_classification_metrics_path, test_classification_metrics_path
from tiu_phi_3_5_mini.utils import FloatLikeT

In [105]:
logger = create_logger(__name__)

In [106]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [107]:
train_valid_split_specs = load_train_validation_splits()
data_selector = ActivationsDataSelector(dsets_index_df, train_valid_split_specs)
dset_idxs_for_6way_topics = data_selector.dset_idxs_for_6way_topics
idxs_for_other_dsets = data_selector.idxs_for_other_dsets
del data_selector

In [108]:
logger.info(f"loading probe-training scenarios from file {train_scenarios_spec_path}")
with train_scenarios_spec_path.open("r") as f:
    raw_scenarios_specs = json.load(f)
train_scenarios = [ProbeTrainScenario(**raw_scenario_spec) for raw_scenario_spec in raw_scenarios_specs]

2025-02-28 17:06:37,048;__main__;INFO:loading probe-training scenarios from file train_scenarios_spec.json


In [109]:
with directions_reconstruction_losses_path.open("r") as f:
    raw_dir_recon_losses = json.load(f)
directions_reconstruction_losses: dict[tuple[int,...], list[ReconLosses]] = {
    tuple(map(int, scenario_id_str.split())) : [ReconLosses(**raw_recon_loss) for raw_recon_loss in raw_recon_losses_lst] for scenario_id_str, raw_recon_losses_lst in raw_dir_recon_losses.items()
}

In [110]:
num_standard_topics = dsets_index_df[dsets_index_df["is_other"] == False]["Categ_Folder"].nunique()
assert num_standard_topics == len(dset_idxs_for_6way_topics)

In [111]:
#'scenarios' describe the different combinations of data that directions/probes were trained on 
other_categs: set[str] = set(dsets_index_df.loc[dsets_index_df.is_other].Categ_Folder.to_list())

scenario_standard_categs: dict[tuple[int,...], list[str]] = {}
scenario_other_dset_names: dict[tuple[int,...], list[str]] = {}
for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    scenario_standard_categs[scenario_id] = []
    scenario_other_dset_names[scenario_id] = []
    for dset_idx_in_scenario in scenario_id:
        if not dsets_index_df.loc[dset_idx_in_scenario, "is_other"]:
            curr_standard_categ = dsets_index_df.loc[dset_idx_in_scenario, "Categ_Folder"]
            if curr_standard_categ not in scenario_standard_categs[scenario_id]:
                scenario_standard_categs[scenario_id].append(curr_standard_categ)
        else:
            curr_other_dset_name = os.path.splitext(dsets_index_df.loc[dset_idx_in_scenario, "Dataset_File"])[0]
            assert curr_other_dset_name not in scenario_other_dset_names[scenario_id]
            scenario_other_dset_names[scenario_id].append(curr_other_dset_name)

scenario_data_variants: dict[tuple[int, ...], set[Literal["affirm", "neg", "conj", "disj", "other"]]] = {}
for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    scenario_data_variants[scenario_id] = set()
    for dset_idx_in_scenario in scenario_id:
        assert not dsets_index_df.at[dset_idx_in_scenario, "in_german"]
        scenario_data_variants[scenario_id].add(
            "other" if dsets_index_df.at[dset_idx_in_scenario, "is_other"] else "neg" if dsets_index_df.at[dset_idx_in_scenario, "is_negated"] else "conj" if dsets_index_df.at[dset_idx_in_scenario, "is_conj"] else "disj" if dsets_index_df.at[dset_idx_in_scenario, "is_disj"] else "affirm")

scenario_labels: dict[tuple[int,...], str] = {scenario.scenario_key(): f"{scenario.result_folder_name}-{scenario.scenario_name}" for scenario in train_scenarios}

In [112]:
t_f_sep_by_layer_df = pd.read_csv(separation_by_layer_analysis_path, index_col="Idx")
ambig_truth_idx, ambig_lie_idx = idxs_for_other_dsets["ambiguous_truthful_reply"], idxs_for_other_dsets["ambiguous_lie"]
t_f_sep_by_layer_df.at[ambig_truth_idx, "Separation after 18"] = t_f_sep_by_layer_df.at[ambig_lie_idx, "Separation after 18"]
t_f_sep_by_layer_df.at[ambig_truth_idx, "Separation after 25"] = t_f_sep_by_layer_df.at[ambig_lie_idx, "Separation after 25"]
unambig_truth_idx, unambig_lie_idx = idxs_for_other_dsets["unambiguous_truthful_reply"], idxs_for_other_dsets["unambiguous_lie"]
t_f_sep_by_layer_df.at[unambig_truth_idx, "Separation after 18"] = t_f_sep_by_layer_df.at[unambig_lie_idx, "Separation after 18"]
t_f_sep_by_layer_df.at[unambig_truth_idx, "Separation after 25"] = t_f_sep_by_layer_df.at[unambig_lie_idx, "Separation after 25"]

In [113]:
with train_split_classification_metrics_path.open("r") as f:
    serialized_train_metrics = json.load(f)
with validation_split_classification_metrics_path.open("r") as f:
    serialized_validation_metrics = json.load(f)
train_metrics: dict[tuple[int,...], list[MetricsForDatasetProbes]] = {
    tuple(map(int, scenario_str.split(' '))): [MetricsForDatasetProbes.from_dict(metrics_dict) for metrics_dict in metrics_dicts_lst] 
    for scenario_str, metrics_dicts_lst in serialized_train_metrics.items()
}
validation_metrics: dict[tuple[int,...], list[MetricsForDatasetProbes]] = {
    tuple(map(int, scenario_str.split(' '))): [MetricsForDatasetProbes.from_dict(metrics_dict) for metrics_dict in metrics_dicts_lst] 
    for scenario_str, metrics_dicts_lst in serialized_validation_metrics.items()
}

In [114]:
with test_classification_metrics_path.open("r") as f:
    serialized_probes_metrics_on_test_dsets = json.load(f)
# first level key identifies the scenario (of one or more datasets) which the probe was trained on;
# second level key identifies the previously-unseen dataset which the probe is tested on
# third level index identifies which train-validation split was used to train a particular group of comparable probes 
#  for a scenario
probes_metrics_on_test_dsets: dict[tuple[int,...], dict[int, list[MetricsForDatasetProbes]]] = {
    tuple(map(int, scenario_str.split(' '))): {
        int(test_dset_idx): [MetricsForDatasetProbes.from_dict(serialized_test_metrics) for serialized_test_metrics in serialized_metrics_lst_for_test_dset]
        for test_dset_idx, serialized_metrics_lst_for_test_dset in serialized_test_dsets_metrics_for_scenario.items()
    } for scenario_str, serialized_test_dsets_metrics_for_scenario in serialized_probes_metrics_on_test_dsets.items()
}

In [115]:
class ConfInterval(NamedTuple):
    low_b: FloatLikeT
    up_b: FloatLikeT

    def __str__(self):
        return f"({float(self.low_b):.7},{float(self.up_b):.7})"

    def __repr__(self):
        return self.__str__()

def mean_from_conf_interval(conf_interval: ConfInterval) -> float:
    return float((conf_interval[0]+conf_interval[1])/2)

def calc_conf_interval(vals: Float[np.ndarray, "n"], alpha=0.05) -> ConfInterval:
    """
    calculates a confidence interval for the population mean of a random variable based on a sample of values
    :param vals: a sample of values for some random variable
    :param alpha: chance of incorrectly rejecting the null hypothesis, i.e. complement of confidence interval width
                    based on this parameter's default value, this will by default produce 95% confidence intervals 
    :return: a confidence interval for the population mean of a random variable
    """
    n = len(vals)
    if n == 0:
        return ConfInterval(np.nan, np.nan)

    degrees_of_freedom = n - 1
    t_critical = t.ppf(1-alpha/2, degrees_of_freedom)
    sample_mean, sample_std_dev = np.mean(vals), np.std(vals)
    conf_interval_margin_of_error = t_critical * sample_std_dev / np.sqrt(n)
    return ConfInterval(sample_mean - conf_interval_margin_of_error, sample_mean + conf_interval_margin_of_error)

In [116]:
class TruthPolarityDirsEval(NamedTuple):
    rel_loss_reducts_on_train: Float[np.ndarray, "n"]
    rel_loss_reducts_on_validation: Float[np.ndarray, "n"]
    rel_loss_reducts_on_validation_using_validation_mean: Float[np.ndarray, "n"]

lyr18_dirs_evals: dict[tuple[int,...], TruthPolarityDirsEval] = {}

for scenario_id, recon_losses_lst in directions_reconstruction_losses.items():
    rel_loss_reducts_on_train: list[float] = []
    rel_loss_reducts_on_validation: list[float] = []
    rel_loss_reducts_on_validation_using_validation_mean: list[float] = []
    
    for recon_losses in recon_losses_lst:
        rel_loss_reducts_on_train.append(1-recon_losses.train_mean_activ_and_t_p_dirs_loss_on_train/recon_losses.train_mean_activ_loss_on_train)
        rel_loss_reducts_on_validation.append(1-recon_losses.train_mean_activ_and_t_p_dirs_loss_on_validation/recon_losses.train_mean_activ_loss_on_validation)
        rel_loss_reducts_on_validation_using_validation_mean.append(1-recon_losses.validation_mean_activ_and_t_p_dirs_loss_on_validation/recon_losses.validation_mean_activ_loss_on_validation)
    
    lyr18_dirs_evals[scenario_id] = TruthPolarityDirsEval(
        rel_loss_reducts_on_train= np.asarray(rel_loss_reducts_on_train),
        rel_loss_reducts_on_validation= np.asarray(rel_loss_reducts_on_validation),
        rel_loss_reducts_on_validation_using_validation_mean= np.asarray(rel_loss_reducts_on_validation_using_validation_mean)
    )

In [117]:
@dataclass(frozen=True)
class ProbeEvalOnData:
    accs: Float[np.ndarray, "n"]
    f1s: Float[np.ndarray, "n"]
    briers: Float[np.ndarray, "n"]
    soft_f1s: Float[np.ndarray, "n"]

    @classmethod
    def combine(cls, *probe_evals_on_dsets: 'ProbeEvalOnData') -> 'ProbeEvalOnData':
        accs_lst: list[np.ndarray] = []
        f1s_lst: list[np.ndarray] = []
        briers_lst: list[np.ndarray] = []
        soft_f1s_lst: list[np.ndarray] = []
        
        for probe_eval_on_dset in probe_evals_on_dsets:
            accs_lst.append(probe_eval_on_dset.accs)
            f1s_lst.append(probe_eval_on_dset.f1s)
            briers_lst.append(probe_eval_on_dset.briers)
            soft_f1s_lst.append(probe_eval_on_dset.soft_f1s)
        
        return cls(np.concat(accs_lst), np.concat(f1s_lst), np.concat(briers_lst), np.concat(soft_f1s_lst))


def collect_probe_evals(probes_metrics_lst: list[MetricsForDatasetProbes]) -> tuple[ProbeEvalOnData, ProbeEvalOnData]:
    ttpd_accs: list[float] = []
    ttpd_f1s: list[float] = []
    ttpd_briers: list[float] = []
    ttpd_soft_f1s: list[float] = []
    
    baseline_accs: list[float] = []
    baseline_f1s: list[float] = []
    baseline_briers: list[float] = []
    baseline_soft_f1s: list[float] = []

    for probes_metrics in probes_metrics_lst:        
        ttpd_metrics = probes_metrics.lyr18_probe_metrics
        ttpd_trad_metrics = ttpd_metrics.get_traditional_metrics()
        ttpd_soft_metrics = ttpd_metrics.get_soft_metrics()
        
        ttpd_accs.append(ttpd_trad_metrics.accuracy)
        ttpd_f1s.append(ttpd_trad_metrics.f1)
        ttpd_briers.append(ttpd_metrics.get_brier_score())
        ttpd_soft_f1s.append(ttpd_soft_metrics.soft_f1)
        
        baseline_metrics = probes_metrics.lyr18_baseline_linear_probe_metrics
        baseline_trad_metrics = baseline_metrics.get_traditional_metrics()
        baseline_soft_metrics = baseline_metrics.get_soft_metrics()
        
        baseline_accs.append(baseline_trad_metrics.accuracy)
        baseline_f1s.append(baseline_trad_metrics.f1)
        baseline_briers.append(baseline_metrics.get_brier_score())
        baseline_soft_f1s.append(baseline_soft_metrics.soft_f1)
    
    ttpd_probe_eval = ProbeEvalOnData(
        accs=np.asarray(ttpd_accs), f1s=np.asarray(ttpd_f1s), briers=np.asarray(ttpd_briers), soft_f1s=np.asarray(ttpd_soft_f1s))
    
    baseline_probe_eval = ProbeEvalOnData(
        accs=np.asarray(baseline_accs), f1s=np.asarray(baseline_f1s), briers=np.asarray(baseline_briers), 
        soft_f1s=np.asarray(baseline_soft_f1s))
    
    return ttpd_probe_eval, baseline_probe_eval

ttpd_probes_train_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
baseline_probes_train_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_train_lst in train_metrics.items():        
    ttpd_train_eval, baseline_train_eval = collect_probe_evals(probes_metrics_on_train_lst)
    ttpd_probes_train_evals[scenario_id] = ttpd_train_eval
    baseline_probes_train_evals[scenario_id] = baseline_train_eval

ttpd_probes_val_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
baseline_probes_val_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_validation_lst in validation_metrics.items():
    ttpd_val_eval, baseline_val_eval = collect_probe_evals(probes_metrics_on_validation_lst)
    ttpd_probes_val_evals[scenario_id] = ttpd_val_eval
    baseline_probes_val_evals[scenario_id] = baseline_val_eval

In [118]:
lyr18_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

lyr18_baseline_linear_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    lyr18_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "tf_sep": mean([t_f_sep_by_layer_df.at[dset_idx_in_scenario, "Separation after 18"] 
                        for dset_idx_in_scenario in scenario_id]),
        "train_recon_improv": calc_conf_interval(lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_train),
        "val_recon_improv": calc_conf_interval(lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_validation),
        "val_recon_improv_w_val_mean": calc_conf_interval(
            lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_validation_using_validation_mean),
        "val_acc": calc_conf_interval(ttpd_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(ttpd_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(ttpd_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(ttpd_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].soft_f1s),
    }
    lyr18_probe_row_dicts.append(lyr18_probe_row_dict)
    
    lyr18_baseline_linear_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "val_acc": calc_conf_interval(baseline_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(baseline_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(baseline_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(baseline_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(baseline_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(baseline_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(baseline_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(baseline_probes_train_evals[scenario_id].soft_f1s),
    }
    lyr18_baseline_linear_probe_row_dicts.append(lyr18_baseline_linear_probe_row_dict)
    
    lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts.append({
        "scenario": scenario_labels[scenario_id],
        "val_acc": calc_conf_interval(
            ttpd_probes_val_evals[scenario_id].accs - baseline_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].f1s - baseline_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(baseline_probes_val_evals[scenario_id].briers - ttpd_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].soft_f1s - baseline_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(ttpd_probes_train_evals[scenario_id].accs - baseline_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].f1s - baseline_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(baseline_probes_train_evals[scenario_id].briers - ttpd_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].soft_f1s - baseline_probes_train_evals[scenario_id].soft_f1s),
    })

ttpd_eval_except_test_df = pd.DataFrame(lyr18_probe_row_dicts)

lyr18_baseline_linear_probes_eval_except_test_df = pd.DataFrame(lyr18_baseline_linear_probe_row_dicts)

lyr18_ttpd_probe_to_baseline_linear_probe_eval_except_test_df = pd.DataFrame(lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts)

For the "tf_sep" aka "true-false separation" column, and the ambiguous/unambiguous truth/lie (plus "honest reply despite incentive to lie") datasets,  
the value in the "ambiguous lie" and "ambiguous truthful reply" rows represent the separation between ambiguous lies and truths,  
the value in the "unambiguous lie" and "unambiguous truthful reply" rows represent the separation between unambiguous lies and truths,
and the 0.0 values for the "honest reply despite incentive to lie" row should be ignored.

When evaluating the below numbers, please note that brier scores are better if they are lower (unlike all other metrics here, which are better if they are higher)

In [119]:
ttpd_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,cities-cities,1.690080,"(0.05712716419138396, 0.05729233970794716)","(0.05703026412330381, 0.05769836129677501)","(0.05707145342981432, 0.057730720564963076)","(0.9803733503292159, 0.9862933163374504)","(0.9806489195501378, 0.9862406358060354)","(0.010495531432461354, 0.015016481287646676)","(0.973136821435391, 0.9776210832985446)","(0.9811340295603423, 0.9827455691018652)","(0.9812479398683122, 0.9827863997069565)","(0.013081116695755958, 0.014231387472980936)","(0.9731493635627246, 0.9754144052372822)"
1,cities-cities_conj,0.388754,"(0.05845964681335783, 0.05875801534263886)","(0.05819145870403357, 0.059390866977574217)","(0.05821055754219779, 0.05944603401517914)","(0.9639927831084723, 0.9736738835581942)","(0.9647255897370349, 0.9739092845817666)","(0.0204190737548407, 0.026250291969087118)","(0.9503783242067544, 0.9558606000960187)","(0.9710742338476971, 0.9734165841823531)","(0.971262702764373, 0.973617055059171)","(0.020860423257705707, 0.022169137888567128)","(0.9530812796421237, 0.9562567828652242)"
2,cities-neg_cities,0.868476,"(0.06480957162814421, 0.06511193135817073)","(0.064198941816476, 0.06538274906180334)","(0.06420243923758949, 0.06540347339426313)","(0.9856674494962961, 0.9899992171703703)","(0.9855302966945338, 0.9899457438982504)","(0.006375311893660074, 0.00906885158773438)","(0.9810028322797107, 0.984880994459092)","(0.9887825994977575, 0.9901471663885304)","(0.9887867601680843, 0.9901426743008367)","(0.006753547997539655, 0.007426806053827045)","(0.9825083807017152, 0.9837027937022718)"
3,relative_comparison-larger_than,0.456852,"(0.06629158979906673, 0.06651427553786605)","(0.06570403071085568, 0.0666135656267361)","(0.06574368969503275, 0.06662698348393384)","(0.9839487690391008, 0.9895360794457476)","(0.9841042130694797, 0.9897018887017617)","(0.008590383775236953, 0.012239886090078417)","(0.9743435367790598, 0.9800487531122111)","(0.987392063339173, 0.9888705629234537)","(0.9873904707066172, 0.9888777070976)","(0.009127694688988096, 0.010076362838322825)","(0.9766383903299812, 0.9791690486315344)"
4,relative_comparison-smaller_than,0.347493,"(0.06011480809109757, 0.06043919713036832)","(0.060094906808161624, 0.06141867614569717)","(0.06008029851738873, 0.06137597461527439)","(0.9924412124648951, 0.9951850501613675)","(0.9925414933609935, 0.9952029424828006)","(0.004479310739612659, 0.006682291202683321)","(0.9845141212637813, 0.9879400026468228)","(0.9932657381476341, 0.9941711305392352)","(0.9932931873734578, 0.9941734863207763)","(0.00536661567721537, 0.00589145252189418)","(0.9851554861967702, 0.9862079069388678)"
5,true_false-common_claim_true_false,0.016943,"(0.00543991293311057, 0.0055085016522392165)","(0.005263253202959811, 0.005535578015593285)","(0.005267541196495162, 0.005544026442337213)","(0.7431772475819516, 0.755923876013554)","(0.7418422229459684, 0.7555674007880201)","(0.16400788078796405, 0.1710721267008439)","(0.6560896376871462, 0.6703295250767866)","(0.7494764775031773, 0.7523493651934519)","(0.748680509956284, 0.7517841870681009)","(0.16645090714177, 0.16781812485436767)","(0.6613063808080574, 0.6674450792423609)"
6,true_false-counterfact_true_false,0.012648,"(0.005195837874715613, 0.005224333893066147)","(0.005109512398583571, 0.0052229884386188925)","(0.005109966152586853, 0.0052237885251670176)","(0.7100643913136933, 0.7148378455078304)","(0.7185406916191844, 0.7240670169150859)","(0.18564795226535852, 0.18793502965517467)","(0.6237598073708209, 0.6284160499426809)","(0.714856796617389, 0.7160453972741289)","(0.7230159951196278, 0.7247977601226614)","(0.18503338803960487, 0.18555147135006014)","(0.62641300734278, 0.6279241958038955)"
7,animal_class-neg_conj,0.447213,"(0.015345968855569174, 0.016224739327048637)","(0.01550509113619547, 0.018540019095507163)","(0.015445364266734738, 0.01848446149721632)","(0.959451

It is striking that including disjunctive data reduces _training_ accuracy substantially (relative to non-disjunctive-containing scenarios in same topic) for any topic whose activations had substantial (>0.10) truth-falsehood separation.

Meanwhile, note that here brier score deltas are calculated so that they are better (for TTPD probes) if they are higher (ideally greater than zero), as with the comparisons of values for other metrics

In [120]:
lyr18_ttpd_probe_to_baseline_linear_probe_eval_except_test_df

,scenario,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,cities-cities,"(-0.009546698751989826, -0.004786634581343516)","(-0.00941672508531746, -0.004731892904050556)","(-0.008133209563691466, -0.004986786114391193)","(-0.01576178537034118, -0.010506391207778382)","(-0.016755089437535488, -0.013178020930357491)","(-0.01668590883603833, -0.013133553221158258)","(-0.012519246061179505, -0.01012667330848802)","(-0.021700496475047987, -0.01598492590507007)"
1,cities-cities_conj,"(-0.012685430974688638, -0.0056479023586446745)","(-0.012395249446682206, -0.005550235626911319)","(-0.009615537416501996, -0.005976818470593139)","(-0.02042325722042017, -0.014195329604087118)","(-0.024038195741245782, -0.018532755844730877)","(-0.02381835399709996, -0.01835469624068886)","(-0.017717958658766863, -0.013966364521333098)","(-0.03133885383971039, -0.023298471193968755)"
2,cities-neg_cities,"(-0.003934055670559416, -0.0013992776627739182)","(-0.004027487017349595, -0.0013891950297185386)","(-0.0018444107061197912, -0.0008711660419334422)","(-0.005046286803425133, -0.0033304230908396673)","(-0.007546659441996968, -0.0054968188188726)","(-0.007537038278088354, -0.005495815790899111)","(-0.0044183853964833136, -0.0031712413605647326)","(-0.008761383712586526, -0.006009899963025064)"
3,relative_comparison-larger_than,"(-0.01573293366636332, -0.010277167343737674)","(-0.015581136344477452, -0.010111496156911648)","(-0.012108307692798224, -0.008458695591768435)","(-0.025415297626131775, -0.019677397186274093)","(-0.012607936660827221, -0.011129437076546513)","(-0.012609529293382757, -0.011122292902400046)","(-0.010076362519296044, -0.009127693691640024)","(-0.023360863930993455, -0.020829700336593242)"
4,relative_comparison-smaller_than,"(-0.007376663602704941, -0.004744548518507147)","(-0.007277463938084124, -0.0047287236595183225)","(-0.0065322801720208625, -0.004433476780879233)","(-0.015116839017364473, -0.011866417114523539)","(-0.0067342618523662506, -0.005828869460765076)","(-0.006706812626542182, -0.005826513679223787)","(-0.005891452130394289, -0.005366614757487851)","(-0.014843682164907886, -0.01379109963663188)"
5,true_false-common_claim_true_false,"(-0.05317133417220266, -0.04345787931094342)","(-0.05656695570384728, -0.047742040647785054)","(-0.030239547082111885, -0.025175919740823068)","(-0.06733303022600629, -0.05595093298870547)","(-0.10370421853399066, -0.09873960169072839)","(-0.10614562636530733, -0.10039787848108364)","(-0.059862906738759256, -0.056636414159403944)","(-0.09772375473436827, -0.0876588707432835)"
6,true_false-counterfact_true_false,"(-0.1114722591645976, -0.10722006056010128)","(-0.10306045492081597, -0.09860203387977323)","(-0.06832733453284981, -0.06639236850538535)","(-0.1405968355024378, -0.13734563903306535)","(-0.1285667799483317, -0.12707046536862887)","(-0.12065077641663605, -0.11851133203436223)","(-0.07854105226013822, -0.07793018568091314)","(-0.151126444111814, -0.14888573153611703)"
7,animal_class-neg_conj,"(-0.026656242768567116, -0.014697140690079491)","(-0.026452226521792438, -0.014859175038326523)","(-0.019056395896960763, -0.012652890975332601)","(-0.04061858689423578, -0.028802654750160236)","(-0.030863864212617078, -0.027893197934275558)","(-0.031688580566895205, -0.028665817391871497)","(-0.02308044462404624, -0.021568254201363084)","(-0.04762351821513272, -0.04157586994736498)"
8,animal_class-neg_disj,"(-0.0707975963091188, -0.03596932098411426)","(-0.09757171663444367, -0.05519524340640172)","(-0.0526490453047717, -0.039425629287210565)","(-0.13296557890793895, -0.11039024389670346)","(-0.1091128120658249, -0.08843897701138795)","(-0.1310726095347525, -0.1069106000367545)","(-0.07945349750368952, -0.06788809025980763)","(-0.1673968597261556, -0.1398837348871853)"
9,animal_class-affirm_neg_conj_disj,"(-0.17974171335803232, -0.14958159491264436)","(-0.21271690563747286, -0.17817680019229765)","(-0.08321634972087927, -0.07057448309981229)","(-0.163572045067

The simple linear probes seem to be doing at least slightly better at generalizing within topic and data variant than the TTPD probes on Phi 3.5 mini (3.8B) for every single training scenario. Meanwhile, Bürger et al found the TTPD probes generalizing (to unseen datasets) only slightly worse than the LR baseline on average (and the difference was not statistically significant). It is puzzling that the TTPD probes dramatically (by 10-20pp of validation accuracy) underperform the LR probes on several of my scenarios (while slightly underperforming on the remaining scenarios, so the average accuracy delta would be nontrivial).

In [121]:
# Specifically standard scenarios containing datasets from within a single topic
def collect_probe_evals_of_scenarios_with_suffix(evals_dict: dict[tuple[int,...], ProbeEvalOnData], scenario_name_suffix: str) -> ProbeEvalOnData:
    scenarios_keys =  [key for key, label in scenario_labels.items() if label.endswith(scenario_name_suffix) and not label.startswith("cross_topic-")]
    scenarios_probe_evals = [evals_dict[scenario_key] for scenario_key in scenarios_keys]
    return ProbeEvalOnData.combine(*scenarios_probe_evals)
    

In [122]:
ttpd_affirm_neg_conj_disj_train_accs = collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-affirm_neg_conj_disj").accs
ttpd_affirm_neg_conj_disj_val_accs = collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-affirm_neg_conj_disj").accs

baseline_affirm_neg_conj_disj_train_accs = collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-affirm_neg_conj_disj").accs
baseline_affirm_neg_conj_disj_val_accs = collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-affirm_neg_conj_disj").accs

print("Confidence intervals for accuracy when trained on all four main data variants within a topic")
pd.DataFrame({
    "metric": ["train_acc", "val_acc"],
    "TTPD": [calc_conf_interval(ttpd_affirm_neg_conj_disj_train_accs), calc_conf_interval(ttpd_affirm_neg_conj_disj_val_accs)],
    "Baseline LR": [calc_conf_interval(baseline_affirm_neg_conj_disj_train_accs), calc_conf_interval(baseline_affirm_neg_conj_disj_val_accs)],
    "TTPD Advantage": [calc_conf_interval(ttpd_affirm_neg_conj_disj_train_accs - baseline_affirm_neg_conj_disj_train_accs), 
                       calc_conf_interval(ttpd_affirm_neg_conj_disj_val_accs - baseline_affirm_neg_conj_disj_val_accs)]
})

Confidence intervals for accuracy when trained on all four main data variants within a topic


,metric,TTPD,Baseline LR,TTPD Advantage
0,train_acc,"(0.7517706664085207, 0.7851710237945463)","(0.9526142614035458, 0.9663045940748747)","(-0.2027984695358333, -0.17917869573952033)"
1,val_acc,"(0.7512174904724367, 0.7843350513577199)","(0.9189312521820524, 0.9398462662807031)","(-0.17296981527025546, -0.15025516136234357)"


In [123]:
ttpd_affirm_neg_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-affirm_neg").accs)
ttpd_affirm_neg_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-affirm_neg").accs)
ttpd_neg_conj_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-neg_conj").accs)
ttpd_neg_conj_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-neg_conj").accs)
ttpd_neg_disj_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-neg_disj").accs)
ttpd_neg_disj_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-neg_disj").accs)

baseline_affirm_neg_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-affirm_neg").accs)
baseline_affirm_neg_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-affirm_neg").accs)
baseline_neg_conj_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-neg_conj").accs)
baseline_neg_conj_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-neg_conj").accs)
baseline_neg_disj_train_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-neg_disj").accs)
baseline_neg_disj_val_acc = calc_conf_interval(
    collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-neg_disj").accs)

print("Confidence intervals for accuracy when trained on 2 of the main data variants within a topic (rather than all 4)")
pd.DataFrame({
    "scenario": ["affirmative+negated", "negated+conjunction", "negated+disjunction"],
    "ttpd_train_acc": [ttpd_affirm_neg_train_acc, ttpd_neg_conj_train_acc, ttpd_neg_disj_train_acc],
    "ttpd_val_acc": [ttpd_affirm_neg_val_acc, ttpd_neg_conj_val_acc, ttpd_neg_disj_val_acc],
    "baseline_train_acc": [baseline_affirm_neg_train_acc, baseline_neg_conj_train_acc, baseline_neg_disj_train_acc],
    "baseline_val_acc": [baseline_affirm_neg_val_acc, baseline_neg_conj_val_acc, baseline_neg_disj_val_acc],
})

Confidence intervals for accuracy when trained on 2 of the main data variants within a topic (rather than all 4)


,scenario,ttpd_train_acc,ttpd_val_acc,baseline_train_acc,baseline_val_acc
0,affirmative+negated,"(0.8996212420970882, 0.9326108625653371)","(0.8967499141091512, 0.9316509889348413)","(0.9521251299727275, 0.9700356389668002)","(0.9308975085330653, 0.9543252917190554)"
1,negated+conjunction,"(0.8842750830326866, 0.9230697156571761)","(0.8776264694655698, 0.9175929708373136)","(0.9746047438995313, 0.9845087093154712)","(0.940641104861379, 0.9609452093593585)"
2,negated+disjunction,"(0.7950971254428607, 0.8209908022622123)","(0.7877190795898098, 0.8159491591334737)","(0.9365962695794252, 0.9553932698679266)","(0.8964163727462908, 0.923134952845653)"


One interesting pattern here is that, while the TTPD probes have lower training accuracy than LR probes and likewise have lower validation accuracy than LR probes, the train-validation generalization gap is consistently much smaller for TTPD probes than for LR probes.

This aggregate analysis also confirms that activations for disjunctive statements are markedly more difficult to classify than activations for affirmative or conjunctive statements. Meanwhile, activations for conjunctive statements are only modestly/not-statistically-significantly harder to classify than activations for affirmative statements.

In [124]:
ttpd_eval_except_test_df['tf_sep'].describe()

count    32.000000
mean      0.384644
std       0.375038
min       0.012648
25%       0.059220
50%       0.353120
75%       0.463822
max       1.690080
Name: tf_sep, dtype: float64

Truth-falsehood separation in the activations data varied dramatically between datasets.

In [125]:
t_p_dirs_stats_means = ttpd_eval_except_test_df[['train_recon_improv', 'val_recon_improv', 'val_recon_improv_w_val_mean', "val_acc",
                                                 "train_acc", "val_f1", "train_f1"]].map(mean_from_conf_interval)
t_p_dirs_stats_means.insert(0, 'tf_sep', ttpd_eval_except_test_df['tf_sep'])
(t_f_sep_and_t_p_dirs_correlations:= t_p_dirs_stats_means.corr())

,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,train_acc,val_f1,train_f1
tf_sep,1.000000,0.581042,0.582961,0.582810,0.676543,0.663971,0.675718,0.660950
train_recon_improv,0.581042,1.000000,0.999426,0.999424,0.695648,0.684698,0.711383,0.699573
val_recon_improv,0.582961,0.999426,1.000000,0.999999,0.692288,0.680788,0.708687,0.696020
val_recon_improv_w_val_mean,0.582810,0.999424,0.999999,1.000000,0.691783,0.680282,0.708215,0.695543
val_acc,0.676543,0.695648,0.692288,0.691783,1.000000,0.998707,0.992270,0.992715
train_acc,0.663971,0.684698,0.680788,0.680282,0.998707,1.000000,0.988982,0.992309
val_f1,0.675718,0.711383,0.708687,0.708215,0.992270,0.988982,1.000000,0.998151
train_f1,0.660950,0.699573,0.696020,0.695543,0.992715,0.992309,0.998151,1.000000


The different reconstruction improvement metrics were extremely highly correlated, indicating that the learned directions generalized well from the train-split data they were calculated with to the validation-split data that they were evaluated on (and that there wasn't a significant shift in the mean of the activations between those two populations).

Meanwhile, train accuracy/F1 were very highly correlated with each other (and so were validation accuracy/F1), which makes sense because these datasets were constructed to be balanced between true and false statements.
Also, validation accuracy and train accuracy were very highly correlated, which would at minimum suggest a very consistent level of generalization loss between scenarios.

Truth-falsehood separation in the activation vectors correlated only fairly strongly with the 'reconstruction improvement' metrics and a bit more strongly than that with the train/validation accuracy/F1 scores.

In [126]:
def collect_accs_for_selected_test_dsets(test_dset_idxs: list[int] | set[int], metrics_on_dsets_and_splits: dict[int, list[MetricsForDatasetProbes]]
                                         ) -> tuple[Float[np.ndarray, "n"], Float[np.ndarray, "n"]]:
    selected_dset_idxs: list[int] = list(test_dset_idxs) if isinstance(test_dset_idxs, set) else test_dset_idxs
    assert len(selected_dset_idxs) > 0
    assert all(dset_idx in metrics_on_dsets_and_splits for dset_idx in selected_dset_idxs)
    n_splits = len(metrics_on_dsets_and_splits[selected_dset_idxs[0]])
    assert all(len(metrics_on_dsets_and_splits[dset_idx]) == n_splits for dset_idx in selected_dset_idxs[1:])
    dsets_metrics_lists_to_merge = [metrics_on_dsets_and_splits[dset_idx] for dset_idx in selected_dset_idxs]
    relevant_test_metrics = [MetricsForDatasetProbes.combine(*[dset_metrics_lst[split_idx] for dset_metrics_lst in dsets_metrics_lists_to_merge])
                             for split_idx in range(n_splits)]
    ttpd_metrics, baseline_metrics = collect_probe_evals(relevant_test_metrics)
    
    return ttpd_metrics.accs, baseline_metrics.accs


In [127]:
# Just using accuracy because there are going to be _so_ many columns in this analysis's dataframe and examination of validation accuracy vs f1 vs soft-f1 for layer 18 probes showed that they were generally very similar for the standard topics' datasets (because those datasets were specifically constructed to be balanced between true and false statements)

@dataclass
class GeneralizationWithinVsAcrossTopicsAccuraciesCollector:
    affirm_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    affirm_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    neg_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    neg_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    conj_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    conj_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    disj_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    disj_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    de_affirm_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    de_affirm_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    de_neg_within_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)
    de_neg_across_topics: list[Float[np.ndarray, "_n"]] = field(default_factory=list)

class GeneralizationWithinVsAcrossTopicsAccuracies(NamedTuple):
    # the different ndarrays may have very different lengths
    affirm_within_topics: Float[np.ndarray, "_n"]
    affirm_across_topics: Float[np.ndarray, "_n"]
    neg_within_topics: Float[np.ndarray, "_n"]
    neg_across_topics: Float[np.ndarray, "_n"]
    conj_within_topics: Float[np.ndarray, "_n"]
    conj_across_topics: Float[np.ndarray, "_n"]
    disj_within_topics: Float[np.ndarray, "_n"]
    disj_across_topics: Float[np.ndarray, "_n"]
    de_affirm_within_topics: Float[np.ndarray, "_n"]
    de_affirm_across_topics: Float[np.ndarray, "_n"]
    de_neg_within_topics: Float[np.ndarray, "_n"]
    de_neg_across_topics: Float[np.ndarray, "_n"]

def safely_concat_ndarrays(ndarrays: list[Float[np.ndarray, "_n"]]) -> Float[np.ndarray, "_n_final"]:
    return np.concat(ndarrays) if ndarrays else np.array([])

def finalize_topic_generalization_accuracies_collection(
        collector: GeneralizationWithinVsAcrossTopicsAccuraciesCollector
) -> GeneralizationWithinVsAcrossTopicsAccuracies:
    return GeneralizationWithinVsAcrossTopicsAccuracies(
        affirm_within_topics=safely_concat_ndarrays(collector.affirm_within_topics),
        affirm_across_topics=safely_concat_ndarrays(collector.affirm_across_topics),
        neg_within_topics=safely_concat_ndarrays(collector.neg_within_topics),
        neg_across_topics=safely_concat_ndarrays(collector.neg_across_topics),
        conj_within_topics=safely_concat_ndarrays(collector.conj_within_topics),
        conj_across_topics=safely_concat_ndarrays(collector.conj_across_topics),
        disj_within_topics=safely_concat_ndarrays(collector.disj_within_topics),
        disj_across_topics=safely_concat_ndarrays(collector.disj_across_topics),
        de_affirm_within_topics=safely_concat_ndarrays(collector.de_affirm_within_topics),
        de_affirm_across_topics=safely_concat_ndarrays(collector.de_affirm_across_topics),
        de_neg_within_topics=safely_concat_ndarrays(collector.de_neg_within_topics),
        de_neg_across_topics=safely_concat_ndarrays(collector.de_neg_across_topics)
    )

class MultiDataVariantScenarioTypes(Enum):
    AFFIRM_NEG = auto()
    NEG_CONJ = auto()
    NEG_DISJ = auto()
    AFFIRM_NEG_CONJ_DISJ = auto()
    CROSSTOPIC_AFFIRM_NEG = auto()
    CROSSTOPIC_AFFIRM_NEG_CONJ = auto()

non_affirm_scenario_types = {MultiDataVariantScenarioTypes.NEG_CONJ, MultiDataVariantScenarioTypes.NEG_DISJ}
non_neg_scenario_types = set()
non_conj_scenario_types = {MultiDataVariantScenarioTypes.AFFIRM_NEG, MultiDataVariantScenarioTypes.NEG_DISJ,
                           MultiDataVariantScenarioTypes.CROSSTOPIC_AFFIRM_NEG}
non_disj_scenario_types = set(MultiDataVariantScenarioTypes) - {MultiDataVariantScenarioTypes.NEG_DISJ,
                                                                MultiDataVariantScenarioTypes.AFFIRM_NEG_CONJ_DISJ}

single_topic_multi_variant_scenario_types = (MultiDataVariantScenarioTypes.AFFIRM_NEG, MultiDataVariantScenarioTypes.NEG_CONJ,
         MultiDataVariantScenarioTypes.NEG_DISJ, MultiDataVariantScenarioTypes.AFFIRM_NEG_CONJ_DISJ)

ttpd_topic_accuracies_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, GeneralizationWithinVsAcrossTopicsAccuraciesCollector] = {
    variant: GeneralizationWithinVsAcrossTopicsAccuraciesCollector()
    for variant in MultiDataVariantScenarioTypes
}

# i.e. ttpd - baseline
ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, GeneralizationWithinVsAcrossTopicsAccuraciesCollector] = {
    variant: GeneralizationWithinVsAcrossTopicsAccuraciesCollector()
    for variant in MultiDataVariantScenarioTypes
}

standard_categ_names_set = set(dset_idxs_for_6way_topics.keys())
for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items():
    curr_standard_categs = scenario_standard_categs[scenario_id]
    curr_data_variants = scenario_data_variants[scenario_id]
    if not curr_standard_categs or len(curr_data_variants) == 1 or "other" in curr_data_variants:
        # about the middle condition: there are only 3 scenarios with a single data variant from a single category (all of them from 'cities') because most standard-topic data files are too small for single-data-variant scenarios to make sense; given that, it doesn't make sense for this aggregating analysis to consider them
        continue

    unseen_standard_categs = list(standard_categ_names_set - set(curr_standard_categs))
    
    scenario_type: MultiDataVariantScenarioTypes
    if len(curr_standard_categs) > 1:
        if curr_data_variants == {"affirm", "neg"}:
            scenario_type = MultiDataVariantScenarioTypes.CROSSTOPIC_AFFIRM_NEG
        elif curr_data_variants == {"affirm", "neg", "conj"}:
            scenario_type = MultiDataVariantScenarioTypes.CROSSTOPIC_AFFIRM_NEG_CONJ
        else:
            raise ValueError(f"this analysis didn't expect a cross-topic ({curr_standard_categs}) scenario with data variants {curr_data_variants}")
    elif curr_data_variants == {"affirm", "neg"}:
        scenario_type = MultiDataVariantScenarioTypes.AFFIRM_NEG
    elif curr_data_variants == {"neg", "conj"}:
        scenario_type = MultiDataVariantScenarioTypes.NEG_CONJ
    elif curr_data_variants == {"neg", "disj"}:
        scenario_type = MultiDataVariantScenarioTypes.NEG_DISJ
    elif curr_data_variants == {"affirm", "neg", "conj", "disj"}:
        scenario_type = MultiDataVariantScenarioTypes.AFFIRM_NEG_CONJ_DISJ
    else:
        raise ValueError(f"this analysis didn't expect a single-topic ({curr_standard_categs}) scenario with data variants {curr_data_variants}")

    ttpd_acc_aggregator_for_curr_scenario_type = ttpd_topic_accuracies_aggregator_by_src_variants_selection[scenario_type]
    # adv- advantage relative to the baseline probe
    ttpd_acc_adv_aggregator_for_curr_scenario_type = ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection[scenario_type]


    diff_topics_affirm_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["affirm"] for topic_nm in unseen_standard_categs]
    across_topics_affirm_ttpd_accs, across_topics_affirm_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_affirm_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.affirm_across_topics.append(across_topics_affirm_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.affirm_across_topics.append(
        across_topics_affirm_ttpd_accs - across_topics_affirm_baseline_accs)

    if "affirm" not in curr_data_variants:
        same_topics_affirm_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["affirm"] for topic_nm in
                                        curr_standard_categs]
        within_topics_affirm_ttpd_accs, within_topics_affirm_baseline_accs = collect_accs_for_selected_test_dsets(
            same_topics_affirm_dset_idxs, test_dsets_metrics)
        ttpd_acc_aggregator_for_curr_scenario_type.affirm_within_topics.append(within_topics_affirm_ttpd_accs)
        ttpd_acc_adv_aggregator_for_curr_scenario_type.affirm_within_topics.append(
            within_topics_affirm_ttpd_accs - within_topics_affirm_baseline_accs)
    
    
    diff_topics_neg_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["neg"] for topic_nm in unseen_standard_categs]
    across_topics_neg_ttpd_accs, across_topics_neg_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_neg_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.neg_across_topics.append(across_topics_neg_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.neg_across_topics.append(
        across_topics_neg_ttpd_accs - across_topics_neg_baseline_accs)

    if "neg" not in curr_data_variants:
        same_topics_neg_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["neg"] for topic_nm in curr_standard_categs]
        within_topics_neg_ttpd_accs, within_topics_neg_baseline_accs = collect_accs_for_selected_test_dsets(
            same_topics_neg_dset_idxs, test_dsets_metrics)
        ttpd_acc_aggregator_for_curr_scenario_type.neg_within_topics.append(within_topics_neg_ttpd_accs)
        ttpd_acc_adv_aggregator_for_curr_scenario_type.neg_within_topics.append(
            within_topics_neg_ttpd_accs - within_topics_neg_baseline_accs)
    
    
    diff_topics_conj_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["conj"] for topic_nm in unseen_standard_categs]
    across_topics_conj_ttpd_accs, across_topics_conj_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_conj_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.conj_across_topics.append(across_topics_conj_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.conj_across_topics.append(
        across_topics_conj_ttpd_accs - across_topics_conj_baseline_accs)

    if "conj" not in curr_data_variants:
        same_topics_conj_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["conj"] for topic_nm in curr_standard_categs]
        within_topics_conj_ttpd_accs, within_topics_conj_baseline_accs = collect_accs_for_selected_test_dsets(
            same_topics_conj_dset_idxs, test_dsets_metrics)
        ttpd_acc_aggregator_for_curr_scenario_type.conj_within_topics.append(within_topics_conj_ttpd_accs)
        ttpd_acc_adv_aggregator_for_curr_scenario_type.conj_within_topics.append(
            within_topics_conj_ttpd_accs - within_topics_conj_baseline_accs)
    
    
    diff_topics_disj_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["disj"] for topic_nm in unseen_standard_categs]
    across_topics_disj_ttpd_accs, across_topics_disj_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_disj_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.disj_across_topics.append(across_topics_disj_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.disj_across_topics.append(
        across_topics_disj_ttpd_accs - across_topics_disj_baseline_accs)

    if "disj" not in curr_data_variants:
        same_topics_disj_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["disj"] for topic_nm in curr_standard_categs]
        within_topics_disj_ttpd_accs, within_topics_disj_baseline_accs = collect_accs_for_selected_test_dsets(
            same_topics_disj_dset_idxs, test_dsets_metrics)
        ttpd_acc_aggregator_for_curr_scenario_type.disj_within_topics.append(within_topics_disj_ttpd_accs)
        ttpd_acc_adv_aggregator_for_curr_scenario_type.disj_within_topics.append(
            within_topics_disj_ttpd_accs - within_topics_disj_baseline_accs)
    
    
    diff_topics_de_affirm_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["de_affirm"] for topic_nm in unseen_standard_categs]
    across_topics_de_affirm_ttpd_accs, across_topics_de_affirm_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_de_affirm_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.de_affirm_across_topics.append(across_topics_de_affirm_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.de_affirm_across_topics.append(
        across_topics_de_affirm_ttpd_accs - across_topics_de_affirm_baseline_accs)
    
    assert "de_affirm" not in curr_data_variants
    same_topics_de_affirm_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["de_affirm"] for topic_nm in curr_standard_categs]
    within_topics_de_affirm_ttpd_accs, within_topics_de_affirm_baseline_accs = collect_accs_for_selected_test_dsets(
        same_topics_de_affirm_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.de_affirm_within_topics.append(within_topics_de_affirm_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.de_affirm_within_topics.append(
        within_topics_de_affirm_ttpd_accs - within_topics_de_affirm_baseline_accs)
    
    
    diff_topics_de_neg_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["de_neg"] for topic_nm in unseen_standard_categs]
    across_topics_de_neg_ttpd_accs, across_topics_de_neg_baseline_accs = collect_accs_for_selected_test_dsets(
        diff_topics_de_neg_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.de_neg_across_topics.append(across_topics_de_neg_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.de_neg_across_topics.append(
        across_topics_de_neg_ttpd_accs - across_topics_de_neg_baseline_accs)

    
    assert "de_neg" not in curr_data_variants
    same_topics_de_neg_dset_idxs = [dset_idxs_for_6way_topics[topic_nm]["de_neg"] for topic_nm in curr_standard_categs]
    within_topics_de_neg_ttpd_accs, within_topics_de_neg_baseline_accs = collect_accs_for_selected_test_dsets(
        same_topics_de_neg_dset_idxs, test_dsets_metrics)
    ttpd_acc_aggregator_for_curr_scenario_type.de_neg_within_topics.append(within_topics_de_neg_ttpd_accs)
    ttpd_acc_adv_aggregator_for_curr_scenario_type.de_neg_within_topics.append(
        within_topics_de_neg_ttpd_accs - within_topics_de_neg_baseline_accs)


ttpd_topic_accuracies_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, GeneralizationWithinVsAcrossTopicsAccuracies] = {
    scenario_type: finalize_topic_generalization_accuracies_collection(accuracies_collector) 
    for scenario_type, accuracies_collector in ttpd_topic_accuracies_aggregator_by_src_variants_selection.items()
}

ttpd_topic_accuracy_advantages_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, GeneralizationWithinVsAcrossTopicsAccuracies] = {
    scenario_type: finalize_topic_generalization_accuracies_collection(accuracies_collector) 
    for scenario_type, accuracies_collector in ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection.items()
}

ttpd_topic_accuracies_by_src_variants_selection_row_dicts: list[dict[str, str | ConfInterval]] = [
    {
        "source_data_variants": scenario_type.name,
        "all_within_topics": calc_conf_interval(np.concat([
            accuracies.affirm_within_topics, accuracies.neg_within_topics, accuracies.conj_within_topics,
            accuracies.disj_within_topics, accuracies.de_affirm_within_topics, accuracies.de_neg_within_topics
        ])),
        "all_across_topics_unseen_variants": calc_conf_interval(np.concat([
            accuracies.affirm_across_topics if scenario_type in non_affirm_scenario_types else np.array([]),
            accuracies.neg_across_topics if scenario_type in non_neg_scenario_types else np.array([]),
            accuracies.conj_across_topics if scenario_type in non_conj_scenario_types else np.array([]),
            accuracies.disj_across_topics if scenario_type in non_disj_scenario_types else np.array([]),
            accuracies.de_affirm_across_topics,
            accuracies.de_neg_across_topics
        ])),
        "affirm_within_topics": calc_conf_interval(accuracies.affirm_within_topics),
        "affirm_across_topics": calc_conf_interval(accuracies.affirm_across_topics),
        "neg_within_topics": calc_conf_interval(accuracies.neg_within_topics),
        "neg_across_topics": calc_conf_interval(accuracies.neg_across_topics),
        "conj_within_topics": calc_conf_interval(accuracies.conj_within_topics),
        "conj_across_topics": calc_conf_interval(accuracies.conj_across_topics),
        "disj_within_topics": calc_conf_interval(accuracies.disj_within_topics),
        "disj_across_topics": calc_conf_interval(accuracies.disj_across_topics),
        "de_affirm_within_topics": calc_conf_interval(accuracies.de_affirm_within_topics),
        "de_affirm_across_topics": calc_conf_interval(accuracies.de_affirm_across_topics),
        "de_neg_within_topics": calc_conf_interval(accuracies.de_neg_within_topics),
        "de_neg_across_topics": calc_conf_interval(accuracies.de_neg_across_topics)
    }
    for scenario_type, accuracies in ttpd_topic_accuracies_by_src_variants_selection.items()
]

within_vs_across_topic_acc_by_source_and_target_data_variants = pd.DataFrame(ttpd_topic_accuracies_by_src_variants_selection_row_dicts)

ttpd_topic_accuracy_advantages_by_src_variants_selection_row_dicts: list[dict[str, str | ConfInterval]] = [
    {
        "source_data_variants": scenario_type.name,
        "all_within_topics": calc_conf_interval(np.concat([
            accuracy_advantages.affirm_within_topics, accuracy_advantages.neg_within_topics,
            accuracy_advantages.conj_within_topics, accuracy_advantages.disj_within_topics,
            accuracy_advantages.de_affirm_within_topics, accuracy_advantages.de_neg_within_topics
        ])),
        "all_across_topics_unseen_variants": calc_conf_interval(safely_concat_ndarrays([
            accuracy_advantages.affirm_across_topics if scenario_type in non_affirm_scenario_types else np.array([]),
            accuracy_advantages.neg_across_topics if scenario_type in non_neg_scenario_types else np.array([]),
            accuracy_advantages.conj_across_topics if scenario_type in non_conj_scenario_types else np.array([]),
            accuracy_advantages.disj_across_topics if scenario_type in non_disj_scenario_types else np.array([]),
            accuracy_advantages.de_affirm_across_topics,
            accuracy_advantages.de_neg_across_topics
        ])),
        "affirm_within_topics": calc_conf_interval(accuracy_advantages.affirm_within_topics),
        "affirm_across_topics": calc_conf_interval(accuracy_advantages.affirm_across_topics),
        "neg_within_topics": calc_conf_interval(accuracy_advantages.neg_within_topics),
        "neg_across_topics": calc_conf_interval(accuracy_advantages.neg_across_topics),
        "conj_within_topics": calc_conf_interval(accuracy_advantages.conj_within_topics),
        "conj_across_topics": calc_conf_interval(accuracy_advantages.conj_across_topics),
        "disj_within_topics": calc_conf_interval(accuracy_advantages.disj_within_topics),
        "disj_across_topics": calc_conf_interval(accuracy_advantages.disj_across_topics),
        "de_affirm_within_topics": calc_conf_interval(accuracy_advantages.de_affirm_within_topics),
        "de_affirm_across_topics": calc_conf_interval(accuracy_advantages.de_affirm_across_topics),
        "de_neg_within_topics": calc_conf_interval(accuracy_advantages.de_neg_within_topics),
        "de_neg_across_topics": calc_conf_interval(accuracy_advantages.de_neg_across_topics)
    }
    for scenario_type, accuracy_advantages in ttpd_topic_accuracy_advantages_by_src_variants_selection.items()
]

within_vs_across_topic_acc_adv_by_source_and_target_data_variants = pd.DataFrame(ttpd_topic_accuracy_advantages_by_src_variants_selection_row_dicts)

within_topic_generalization_acc = calc_conf_interval(
    np.concat([
        one_case_within_topic_accs for one_source_variants_type_accs in ttpd_topic_accuracies_by_src_variants_selection.values() 
        for one_case_within_topic_accs in [
            one_source_variants_type_accs.affirm_within_topics, one_source_variants_type_accs.neg_within_topics,
            one_source_variants_type_accs.conj_within_topics, one_source_variants_type_accs.disj_within_topics,
            one_source_variants_type_accs.de_affirm_within_topics, one_source_variants_type_accs.de_neg_within_topics
        ]
    ])
)

across_topic_unseen_variants_generalization_acc = calc_conf_interval(
    safely_concat_ndarrays([
        one_case_across_topic_accs for src_variants_scenario_type, one_source_variants_type_accs in ttpd_topic_accuracies_by_src_variants_selection.items()
        for one_case_across_topic_accs in [
            one_source_variants_type_accs.affirm_across_topics if src_variants_scenario_type in non_affirm_scenario_types else np.array([]),
            one_source_variants_type_accs.neg_across_topics if src_variants_scenario_type in non_neg_scenario_types else np.array([]),
            one_source_variants_type_accs.conj_across_topics if src_variants_scenario_type in non_conj_scenario_types else np.array([]),
            one_source_variants_type_accs.disj_across_topics if src_variants_scenario_type in non_disj_scenario_types else np.array([]),
            one_source_variants_type_accs.de_affirm_across_topics,
            one_source_variants_type_accs.de_neg_across_topics
        ]
    ])
)

within_topic_generalization_acc_advantage = calc_conf_interval(
    np.concat([
        one_case_within_topic_acc_advs for one_source_variants_type_acc_advs in ttpd_topic_accuracy_advantages_by_src_variants_selection.values() 
        for one_case_within_topic_acc_advs in [
            one_source_variants_type_acc_advs.affirm_within_topics, one_source_variants_type_acc_advs.neg_within_topics,
            one_source_variants_type_acc_advs.conj_within_topics, one_source_variants_type_acc_advs.disj_within_topics,
            one_source_variants_type_acc_advs.de_affirm_within_topics, one_source_variants_type_acc_advs.de_neg_within_topics
        ]
    ])
)

across_topic_unseen_variants_generalization_acc_advantage = calc_conf_interval(
    safely_concat_ndarrays([
        one_case_across_topic_acc_advs for src_variants_scenario_type, one_source_variants_type_acc_advs
        in ttpd_topic_accuracy_advantages_by_src_variants_selection.items()
        for one_case_across_topic_acc_advs in [
            one_source_variants_type_acc_advs.affirm_across_topics if src_variants_scenario_type in non_affirm_scenario_types else np.array([]),
            one_source_variants_type_acc_advs.neg_across_topics if src_variants_scenario_type in non_neg_scenario_types else np.array([]),
            one_source_variants_type_acc_advs.conj_across_topics if src_variants_scenario_type in non_conj_scenario_types else np.array([]),
            one_source_variants_type_acc_advs.disj_across_topics if src_variants_scenario_type in non_disj_scenario_types else np.array([]),
            one_source_variants_type_acc_advs.de_affirm_across_topics, one_source_variants_type_acc_advs.de_neg_across_topics
        ]
    ])
)

The "neg_within_topics" column is hidden because the current selection of multi-data-variant training scenarios doesn't include any scenarios which exclude negated statements. That, in turn, is because the purpose of the Burger et al paper and the TTPD probe design (disentangling the truthfulness and polarity of statements) is undermined by having training scenarios where all data points have the same logical/grammatical polarity.

Aggregate statistics for across-topic generalization are limited to data variants that were unseen by a given scenario in training in order to allow fair comparisons between across-topic and within-topic generalization (since the latter always involves evaluating on unseen data variants).

In [128]:
within_vs_across_topic_acc_by_source_and_target_data_variants

,source_data_variants,all_within_topics,all_across_topics_unseen_variants,affirm_within_topics,affirm_across_topics,neg_within_topics,neg_across_topics,conj_within_topics,conj_across_topics,disj_within_topics,disj_across_topics,de_affirm_within_topics,de_affirm_across_topics,de_neg_within_topics,de_neg_across_topics
0,AFFIRM_NEG,"(0.7150511303929463, 0.7492762203604391)","(0.6900725567215813, 0.7102183786494052)","(nan, nan)","(0.8985324630762255, 0.9166555560690408)","(nan, nan)","(0.828958748897363, 0.8525821195132884)","(0.7057143716878952, 0.7608633986725856)","(0.6992793112344312, 0.7228637876792338)","(0.5648944991028524, 0.5827555008971477)","(0.5637836955664453, 0.5801763044335548)","(0.8293972130372979, 0.8746027869627021)","(0.782051814754012, 0.802548185245988)","(0.7318225516426223, 0.8072590810104389)","(0.7163295911143438, 0.7341310514559373)"
1,NEG_CONJ,"(0.7103181387437059, 0.7445016877692912)","(0.7023632390022998, 0.7272527852870883)","(0.7558840849107619, 0.8323884184201379)","(0.791832594451059, 0.841772091729252)","(nan, nan)","(0.8543547545197634, 0.8687619858427971)","(nan, nan)","(0.8328661684929819, 0.8569014396640602)","(0.5177572138024054, 0.5275761195309279)","(0.525283390549028, 0.5415899427843054)","(0.7536891470409331, 0.8159775196257334)","(0.7361331506558114, 0.7689335160108554)","(0.7866931009576091, 0.8293137017634792)","(0.750170540752523, 0.7627488702247192)"
2,NEG_DISJ,"(0.7069491072573049, 0.737490239925322)","(0.6920097777998357, 0.7160103872140067)","(0.6770645689894971, 0.7516811597499112)","(0.6965908283787902, 0.7662633336432806)","(nan, nan)","(0.8499743911529509, 0.8631516609964593)","(0.6322945081937117, 0.6716219137021493)","(0.6338312298073625, 0.6676177581857765)","(nan, nan)","(0.5718890856274801, 0.5894175810391868)","(0.6679793032096963, 0.7366873634569704)","(0.6477521603648337, 0.6915145063018329)","(0.8017719460981808, 0.8386566253303908)","(0.7548975165447112, 0.7736133268287826)"
3,AFFIRM_NEG_CONJ_DISJ,"(0.762400007199252, 0.801474142460612)","(0.7647820605996511, 0.7743305230683539)","(nan, nan)","(0.8880345286312943, 0.9037169238527243)","(nan, nan)","(0.8596610666640623, 0.8777751243499345)","(nan, nan)","(0.6758408274918327, 0.7146584101677821)","(nan, nan)","(0.539591796973799, 0.5599215363595343)","(0.7372316529381148, 0.7977683470618854)","(0.7788900886891391, 0.7887099113108608)","(0.7716196490231324, 0.8211286502965953)","(0.7479131848661295, 0.762711982469881)"
4,CROSSTOPIC_AFFIRM_NEG,"(0.7159609310956561, 0.7566651315441648)","(0.7409134304288334, 0.7692332736010572)","(nan, nan)","(0.9771133540268268, 0.9780929328101083)","(nan, nan)","(0.9482773459711572, 0.9524593927321826)","(0.634922216283408, 0.6470777837165921)","(0.753726587873664, 0.7631268949125652)","(0.6398793952153173, 0.6656539381180159)","(0.653310209943415, 0.683089790056585)","(0.8242431670763256, 0.8330901662570077)","(0.8339983754307452, 0.8460016245692545)","(0.8139124169321991, 0.8317251669604183)","(0.748043997346881, 0.7592893359864524)"
5,CROSSTOPIC_AFFIRM_NEG_CONJ,"(0.6876664804690278, 0.7531154732968188)","(0.687654287251553, 0.7670346016373359)","(nan, nan)","(0.9752012061827592, 0.9780404441119362)","(nan, nan)","(0.9593528355294778, 0.9647630780265144)","(nan, nan)","(0.8975254941626619, 0.9172863553169218)","(0.5349337889914196, 0.5615328776752471)","(0.51576379221208, 0.5289695411212533)","(0.7661130145604053, 0.7778869854395948)","(0.8866682323259258, 0.8953317676740742)","(0.8322746628640403, 0.8496045317668321)","(0.7662297937559597, 0.771103539577374)"


Generalization to German-language (affirmative or negated) data being stronger than generalization to disjunctive data is consistent with Phi 3.5 mini being trained on multilingual data (including German) and with earlier findings in this notebook that disjunctive statements were not processed by the model in a way that made it easy to classify overall statement truth/falsehood from the final token's activations.

Training on all 4 data variants led to a statistically significant improvement in both within-topic and across-topic generalization relative to training on just 2 variants in a single topic.

Meanwhile, it's peculiar that training on 2-3 variants of multiple topics' data led to a statistically significant improvement in across-topic generalization (relative to training on any 2 variants of a single topic's data) while there was not a statistically significant gain in within-topic generalization.
It's also surprising that there wasn't a statistically-significant improvement in within-topic or across-topic generalization for the multi-topic scenarios when adding conjunctive statements' activations (in addition to affirmative or negated statements' activations).

Surprisingly, within-topic and across-topic generalization to disjunctive data were statistically-significantly worse when a scenario included conjunctive statements' activations. Also, there wasn't a statistically-significant advantage there for scenarios that had trained on negated+disjunctive data vs those that had trained on merely affirmative+negated data.
In fact, the multi-topic scenario that only trained on affirmative or negated statements' activations generalized to unseen topics' disjunctive statements significantly better than the scenarios that had been trained on a single topic's negated or disjunctive statements. This can't be a quirk of which scenarios were picked for that multi-topic scenario, because that multi-topic scenario also generalized to within-topics disjunctive statements better than the disjunctive+negated scenarios generalized to unseen topics' disjunctive statements.

In [129]:
within_vs_across_topic_acc_adv_by_source_and_target_data_variants

,source_data_variants,all_within_topics,all_across_topics_unseen_variants,affirm_within_topics,affirm_across_topics,neg_within_topics,neg_across_topics,conj_within_topics,conj_across_topics,disj_within_topics,disj_across_topics,de_affirm_within_topics,de_affirm_across_topics,de_neg_within_topics,de_neg_across_topics
0,AFFIRM_NEG,"(-0.03891638009339969, -0.011765121432444283)","(-0.008954609875397816, 0.005607021046669526)","(nan, nan)","(0.005481846737780061, 0.04438532831016061)","(nan, nan)","(-0.0575716371842087, -0.044088913481801426)","(0.04794568731940022, 0.08274463310783611)","(0.012250386393996835, 0.04187949925494542)","(-0.04351661425886153, -0.009583385741138463)","(-0.006372741347998848, 0.02081274134799884)","(-0.029961113070603184, -0.009038886929396844)","(-0.006250991649672974, 0.021050991649672994)","(-0.1588569934218773, -0.08245933310873493)","(-0.05936259777005519, -0.037397643193800234)"
1,NEG_CONJ,"(-0.032606340956994335, -0.011905408258895933)","(-0.006988735966352957, 0.007259241588120347)","(-0.014480583437243778, 0.018688008342390126)","(-0.022873134305026074, 0.01670096669838744)","(nan, nan)","(0.003030753451273469, 0.024474189222003832)","(nan, nan)","(-0.026988985516217548, 0.008716521250923098)","(-0.028613712068708587, -0.012586287931291391)","(-0.00907639764479745, 0.0017163976447974582)","(-0.03408680942564782, 0.019420142758981185)","(-0.0325374912865587, 0.003804157953225395)","(-0.08777369736911336, -0.03861405773292742)","(0.014536804985487553, 0.028810718441553945)"
2,NEG_DISJ,"(-0.04656232010420611, -0.02040308862855881)","(0.006977116870323714, 0.024315550067346828)","(-0.07413301617798781, -0.020929527901778086)","(0.018111674051543538, 0.06113328491775809)","(nan, nan)","(0.03779110715634445, 0.08102634048453142)","(-0.0679838700375307, -0.02198528884097397)","(-0.053080557045562325, -0.019810066167702402)","(nan, nan)","(-0.085769079160741, -0.07115092083925897)","(-0.028070916152386473, 0.03873758281905312)","(-0.011833608774609497, 0.014033608774609518)","(-0.06575985674079737, -0.027736741898658376)","(0.046451597378607344, 0.07016473461603791)"
3,AFFIRM_NEG_CONJ_DISJ,"(-0.09815052246529626, -0.0623426748136153)","(0.010049998475824672, 0.02396097876647787)","(nan, nan)","(0.014654464954301488, 0.051075170717387944)","(nan, nan)","(0.029845927738122377, 0.06726613140249381)","(nan, nan)","(-0.17272353329727858, -0.12636165824074314)","(nan, nan)","(-0.1472410541094311, -0.12188561255723551)","(-0.11240118381092236, -0.07293214952241096)","(-0.011092788399698697, 0.002626121733032046)","(-0.09775864096691436, -0.03789442025757542)","(0.02733616806232809, 0.04915245308894364)"
4,CROSSTOPIC_AFFIRM_NEG,"(-0.009442924049357083, 0.010130172371504733)","(-0.02986637973623344, 0.004004063216350863)","(nan, nan)","(0.0005990689772560171, 0.0022496540089915256)","(nan, nan)","(-0.030382847841977683, -0.024184932118729647)","(-0.023935702942023306, -0.0065309637246433565)","(-0.03356957790880706, -0.011346354837389907)","(0.01234720633061931, 0.07318612700271401)","(0.07415922050740925, 0.11130744615925736)","(-0.009398473742008135, 0.00473180707534147)","(-0.031142226292786623, -0.016857773707213405)","(-0.03198464363423268, -0.015666363077176712)","(-0.11984290082593183, -0.07615709917406818)"
5,CROSSTOPIC_AFFIRM_NEG_CONJ,"(-0.024264885211426397, 0.001891581706579248)","(-0.0039412782813094144, 0.01405238939242054)","(nan, nan)","(-0.00017667390577003018, 0.0033692082869095303)","(nan, nan)","(-0.017730673962097217, -0.011296830949494166)","(nan, nan)","(-0.052108898731714585, -0.030437138097748985)","(0.021355020622070335, 0.050844979377929665)","(0.006183154011014131, 0.01948351265565256)","(-0.07852286776295324, -0.06281046557038013)","(0.018174838934598238, 0.03382516106540174)","(-0.00937653978959567, 0.01138996260838759)","(-0.04370523907836238, -0.0036280942549709143)"


In [130]:
within_topic_generalization_acc

(0.7267802,0.7429884)

In [131]:
across_topic_unseen_variants_generalization_acc

(0.7134343,0.7249022)

In [132]:
(avg_within_to_across_topic_generalization_delta := mean_from_conf_interval(across_topic_unseen_variants_generalization_acc) - mean_from_conf_interval(within_topic_generalization_acc))

-0.015716082484992233

In [133]:
within_topic_generalization_acc_advantage

(-0.0391622,-0.02682148)

In [134]:
across_topic_unseen_variants_generalization_acc_advantage

(0.002465391,0.01005233)

The cross-topic generalization difference is (barely) statistically significant but small (a drop of ~0.2-3pp), so we will ignore the "already-seen topic vs fully-unseen topic" distinction for analysis beyond this point.

Intriguingly, while TTPD probes had slightly worse average generalization accuracy than LR probes on unseen datasets in already-seen topics (fitting the pattern of results so far in this notebook), TTPD probes had a very slightly better average generalization accuracy on fully-unseen topics.


In [158]:
collect_true_idxs: Callable[[pd.Series[bool]], set[int]] = lambda bool_series: {idx for idx, is_true in bool_series.to_dict().items() if is_true}

#using more verbose phrasing in just this case to disambiguate from affirmative english texts in 'other' topics (real world scenarios, relative comparison, true false)
en_affirm_in_std_topic_dset_idxs: set[int] = collect_true_idxs(
    ~(dsets_index_df.is_negated | dsets_index_df.is_conj | dsets_index_df.is_disj | dsets_index_df.is_other 
       | dsets_index_df.in_german))
en_neg_dset_idxs: set[int] = collect_true_idxs(dsets_index_df.is_negated & ~dsets_index_df.in_german)
conj_dset_idxs: set[int] = collect_true_idxs(dsets_index_df.is_conj)
disj_dset_idxs: set[int] = collect_true_idxs(dsets_index_df.is_disj)
de_affirm_dset_idxs: set[int] = collect_true_idxs(~dsets_index_df.is_negated & dsets_index_df.in_german)
de_neg_dset_idxs: set[int] = collect_true_idxs(dsets_index_df.is_negated & dsets_index_df.in_german)

In [192]:
# `include_groups=False` averts deprecated behavior warning
categ_to_dset_idxs: dict[str, set[int]] = dsets_index_df.groupby('Categ_Folder').apply(lambda group: set(group.index), include_groups=False).to_dict()

all_categs: list[str] = dsets_index_df.Categ_Folder.unique().tolist()

In [193]:
ttpd_acc_on_multi_topic_unseen_data_groupings_row_dicts: list[dict[str, str | ConfInterval]] = []
ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_row_dicts: list[dict[str, str | ConfInterval]] = []
ttpd_acc_on_single_topic_unseen_data_groupings_row_dicts: list[dict[str, str | ConfInterval]] = []
ttpd_acc_advantage_on_single_topic_unseen_data_groupings_row_dicts: list[dict[str, str | ConfInterval]] = []

In [194]:
all_scenarios_all_unseen_datasets_acc_advantages: list[Float[np.ndarray, "_n"]] = []

In [195]:
CrossTopicGrouping = Literal["affirm", "neg", "conj", "disj", "de_affirm", "de_neg", "all"]
# Note that 'all' here includes unseen datasets in 'other' topics in addition to unseen datasets that belong to standard/6-way topics
cross_topic_variant_groupings = ("affirm", "neg", "conj", "disj", "de_affirm", "de_neg", "all")


ttpd_data_variant_accuracies_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, dict[CrossTopicGrouping, list[Float[np.ndarray, "_n"]]]] = {
    scenario_type: {data_variant: [] for data_variant in cross_topic_variant_groupings}
    for scenario_type in single_topic_multi_variant_scenario_types
}

ttpd_data_variant_accuracy_advantages_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, dict[CrossTopicGrouping, list[Float[np.ndarray, "_n"]]]] = {
    scenario_type: {data_variant: [] for data_variant in cross_topic_variant_groupings}
    for scenario_type in single_topic_multi_variant_scenario_types
}

ttpd_topic_accuracies_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, dict[str, list[Float[np.ndarray, "_n"]]]] = {
    scenario_type: { target_categ : [] for target_categ in all_categs }
    for scenario_type in single_topic_multi_variant_scenario_types
}

ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection: dict[
    MultiDataVariantScenarioTypes, dict[str, list[Float[np.ndarray, "_n"]]]] = {
    scenario_type: { target_categ : [] for target_categ in all_categs }
    for scenario_type in single_topic_multi_variant_scenario_types
}

In [196]:
for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items():
    curr_standard_categs = scenario_standard_categs[scenario_id]
    curr_data_variants = scenario_data_variants[scenario_id]
    curr_scenario_test_dset_idxs = set(test_dsets_metrics.keys())

    ttpd_all_unseen_accs, baseline_all_unseen_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs, test_dsets_metrics)

    ttpd_unseen_en_affirm_accs, baseline_unseen_en_affirm_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & en_affirm_in_std_topic_dset_idxs, test_dsets_metrics)

    ttpd_unseen_en_neg_accs, baseline_unseen_en_neg_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & en_neg_dset_idxs, test_dsets_metrics)

    ttpd_unseen_conj_accs, baseline_unseen_conj_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & conj_dset_idxs, test_dsets_metrics)

    ttpd_unseen_disj_accs, baseline_unseen_disj_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & disj_dset_idxs, test_dsets_metrics)

    ttpd_unseen_de_affirm_accs, baseline_unseen_de_affirm_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & de_affirm_dset_idxs, test_dsets_metrics)

    ttpd_unseen_de_neg_accs, baseline_unseen_de_neg_accs = collect_accs_for_selected_test_dsets(
        curr_scenario_test_dset_idxs & de_neg_dset_idxs, test_dsets_metrics)

    ttpd_categ_accs: dict[str, Float[np.ndarray, "_n"]] = {}
    ttpd_categ_acc_advantages: dict[str, Float[np.ndarray, "_n"]] = {}
    
    for categ_nm, categ_dset_idxs in categ_to_dset_idxs.items():
        unseen_categ_idxs = curr_scenario_test_dset_idxs & categ_dset_idxs
        ttpd_unseen_categ_accs, baseline_unseen_categ_accs = collect_accs_for_selected_test_dsets(unseen_categ_idxs, test_dsets_metrics)
        ttpd_categ_accs[categ_nm] = ttpd_unseen_categ_accs
        ttpd_categ_acc_advantages[categ_nm] = ttpd_unseen_categ_accs - baseline_unseen_categ_accs

    all_scenarios_all_unseen_datasets_acc_advantages.append(ttpd_all_unseen_accs - baseline_all_unseen_accs)

    if len(scenario_id) == 1 or len(curr_standard_categs) > 1:

        ttpd_acc_on_multi_topic_unseen_data_groupings_row_dicts.append({
            "scenario": scenario_labels[scenario_id],
            "all": calc_conf_interval(ttpd_all_unseen_accs),
            "en_affirm": calc_conf_interval(ttpd_unseen_en_affirm_accs),
            "en_negated": calc_conf_interval(ttpd_unseen_en_neg_accs),
            "conj": calc_conf_interval(ttpd_unseen_conj_accs),
            "disj": calc_conf_interval(ttpd_unseen_disj_accs),
            "de_affirm": calc_conf_interval(ttpd_unseen_de_affirm_accs),
            "de_negated": calc_conf_interval(ttpd_unseen_de_neg_accs)
        })
        ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_row_dicts.append({
            "scenario": scenario_labels[scenario_id],
            "all": calc_conf_interval(ttpd_all_unseen_accs - baseline_all_unseen_accs),
            "en_affirm": calc_conf_interval(ttpd_unseen_en_affirm_accs - baseline_unseen_en_affirm_accs),
            "en_negated": calc_conf_interval(ttpd_unseen_en_neg_accs - baseline_unseen_en_neg_accs),
            "conj": calc_conf_interval(ttpd_unseen_conj_accs - baseline_unseen_conj_accs),
            "disj": calc_conf_interval(ttpd_unseen_disj_accs - baseline_unseen_disj_accs),
            "de_affirm": calc_conf_interval(ttpd_unseen_de_affirm_accs - baseline_unseen_de_affirm_accs),
            "de_negated": calc_conf_interval(ttpd_unseen_de_neg_accs - baseline_unseen_de_neg_accs)
        })

        ttpd_acc_on_single_topic_unseen_data_groupings_row_dicts.append({
            "scenario": scenario_labels[scenario_id],
            **{categ_nm: calc_conf_interval(categ_accs) for categ_nm, categ_accs in ttpd_categ_accs.items()}
        })
        ttpd_acc_advantage_on_single_topic_unseen_data_groupings_row_dicts.append({
            "scenario": scenario_labels[scenario_id],
            **{categ_nm: calc_conf_interval(categ_acc_advantages) for categ_nm, categ_acc_advantages in
               ttpd_categ_acc_advantages.items()}
        })
    else:
        scenario_type: MultiDataVariantScenarioTypes
        if curr_data_variants == {"affirm", "neg"}:
            scenario_type = MultiDataVariantScenarioTypes.AFFIRM_NEG
        elif curr_data_variants == {"neg", "conj"}:
            scenario_type = MultiDataVariantScenarioTypes.NEG_CONJ
        elif curr_data_variants == {"neg", "disj"}:
            scenario_type = MultiDataVariantScenarioTypes.NEG_DISJ
        elif curr_data_variants == {"affirm", "neg", "conj", "disj"}:
            scenario_type = MultiDataVariantScenarioTypes.AFFIRM_NEG_CONJ_DISJ
        else:
            raise ValueError(f"this analysis didn't expect a single-topic ({curr_standard_categs}) scenario with data variants {curr_data_variants}")

        variant_accs_aggregator = ttpd_data_variant_accuracies_aggregator_by_src_variants_selection[scenario_type]
        variant_accs_aggregator["all"].append(ttpd_all_unseen_accs)
        variant_accs_aggregator["affirm"].append(ttpd_unseen_en_affirm_accs)
        variant_accs_aggregator["neg"].append(ttpd_unseen_en_neg_accs)
        variant_accs_aggregator["conj"].append(ttpd_unseen_conj_accs)
        variant_accs_aggregator["disj"].append(ttpd_unseen_disj_accs)
        variant_accs_aggregator["de_affirm"].append(ttpd_unseen_de_affirm_accs)
        variant_accs_aggregator["de_neg"].append(ttpd_unseen_de_neg_accs)

        variant_acc_advs_aggregator = ttpd_data_variant_accuracy_advantages_aggregator_by_src_variants_selection[scenario_type]
        variant_acc_advs_aggregator["all"].append(ttpd_all_unseen_accs - baseline_all_unseen_accs)
        variant_acc_advs_aggregator["affirm"].append(ttpd_unseen_en_affirm_accs - baseline_unseen_en_affirm_accs)
        variant_acc_advs_aggregator["neg"].append(ttpd_unseen_en_neg_accs - baseline_unseen_en_neg_accs)
        variant_acc_advs_aggregator["conj"].append(ttpd_unseen_conj_accs - baseline_unseen_conj_accs)
        variant_acc_advs_aggregator["disj"].append(ttpd_unseen_disj_accs - baseline_unseen_disj_accs)
        variant_acc_advs_aggregator["de_affirm"].append(ttpd_unseen_de_affirm_accs - baseline_unseen_de_affirm_accs)
        variant_acc_advs_aggregator["de_neg"].append(ttpd_unseen_de_neg_accs - baseline_unseen_de_neg_accs)

        topic_accs_aggregator = ttpd_topic_accuracies_aggregator_by_src_variants_selection[scenario_type]
        for categ_nm, categ_accs in ttpd_categ_accs.items():
            topic_accs_aggregator[categ_nm].append(categ_accs)

        topic_acc_advs_aggregator = ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection[scenario_type]
        for categ_nm, categ_acc_advantages in ttpd_categ_acc_advantages.items():
            topic_acc_advs_aggregator[categ_nm].append(categ_acc_advantages)

In [197]:
def correct_variant_nm_for_display(variant_name: str) -> str:
    if variant_name == "affirm":
        return "en_affirm"
    elif variant_name == "neg":
        return "en_negated"
    elif variant_name == "de_neg":
        return "de_negated"
    else:
        return variant_name


for scenario_type in single_topic_multi_variant_scenario_types:
    variant_accuracies_aggregator = ttpd_data_variant_accuracies_aggregator_by_src_variants_selection[scenario_type]
    variant_acc_advantages_aggregator = ttpd_data_variant_accuracy_advantages_aggregator_by_src_variants_selection[scenario_type]
    topic_accuracies_aggregator = ttpd_topic_accuracies_aggregator_by_src_variants_selection[scenario_type]
    topic_acc_advantages_aggregator = ttpd_topic_accuracy_advantages_aggregator_by_src_variants_selection[scenario_type]

    ttpd_acc_on_multi_topic_unseen_data_groupings_row_dicts.append({
        "scenario": scenario_type.name,
        **{
            correct_variant_nm_for_display(variant):
                calc_conf_interval(safely_concat_ndarrays(accs_lst))
            for variant, accs_lst in variant_accuracies_aggregator.items()
        }
    })

    ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_row_dicts.append({
        "scenario": scenario_type.name,
        **{
            correct_variant_nm_for_display(variant):
                calc_conf_interval(safely_concat_ndarrays(acc_advs_lst))
            for variant, acc_advs_lst in variant_acc_advantages_aggregator.items()
        }
    })

    ttpd_acc_on_single_topic_unseen_data_groupings_row_dicts.append({
        "scenario": scenario_type.name,
        **{
            topic: calc_conf_interval(safely_concat_ndarrays(accs_lst))
            for topic, accs_lst in topic_accuracies_aggregator.items()
        }
    })

    ttpd_acc_advantage_on_single_topic_unseen_data_groupings_row_dicts.append({
        "scenario": scenario_type.name,
        **{
            topic: calc_conf_interval(safely_concat_ndarrays(acc_advs_lst))
            for topic, acc_advs_lst in topic_acc_advantages_aggregator.items()
        }
    })

In [198]:
ttpd_acc_on_multi_topic_unseen_data_groupings_df = pd.DataFrame(ttpd_acc_on_multi_topic_unseen_data_groupings_row_dicts)
ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_df = pd.DataFrame(ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_row_dicts)
ttpd_acc_on_single_topic_unseen_data_groupings_df = pd.DataFrame(ttpd_acc_on_single_topic_unseen_data_groupings_row_dicts)
ttpd_acc_advantage_on_single_topic_unseen_data_groupings_df = pd.DataFrame(ttpd_acc_advantage_on_single_topic_unseen_data_groupings_row_dicts)

Please note that 'unseen' results for a target data variant aren't completely comparable between scenarios if one scenario was trained on some data of that variant and the other wasn't.

To keep the 'multiple comparisons' problem from getting too egregious (and to make analysis more tractable), the following tables group together scenarios whose source data consists of just 2-4 data variants from within a single topic (grouping based on the choice of data variants).

In [199]:
ttpd_acc_on_multi_topic_unseen_data_groupings_df

,scenario,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
0,cities-cities,"(0.6269713046988542, 0.6409516560262213)","(0.7878284868142649, 0.8027759416716715)","(0.6521527640753247, 0.7008627079802482)","(0.701860123300467, 0.7382099117170419)","(0.6034719647343465, 0.6343613685989868)","(0.7381753862711473, 0.7491579470621861)","(0.5159157314915636, 0.5449538337258276)"
1,cities-cities_conj,"(0.6660072788463463, 0.6774894544139733)","(0.8311892153349667, 0.8523283091361413)","(0.5112599685570582, 0.5116323585664024)","(0.8771084620912775, 0.8820115379087226)","(0.5077734982378647, 0.5088265017621351)","(0.7855005741810817, 0.8008327591522517)","(0.5125752650467247, 0.5175250694014358)"
2,cities-neg_cities,"(0.6860292164900346, 0.6869975961987873)","(0.8864364142113027, 0.8917448298682676)","(0.8101272699898249, 0.8139301806385416)","(0.7479681028729785, 0.7550083853711436)","(0.5315556688106113, 0.5423109978560556)","(0.7905737503319358, 0.7994262496680643)","(0.7896681169559465, 0.7932750268567628)"
3,relative_comparison-larger_than,"(0.6006057568346083, 0.6107232316553682)","(0.6628360141614846, 0.6855378412221593)","(0.5129453655424171, 0.5138939145333644)","(0.7906126884489856, 0.8099375866885831)","(0.54427419703006, 0.5511591363032737)","(0.7583280830547159, 0.7656719169452841)","(0.5183946488294315, 0.5183946488294315)"
4,relative_comparison-smaller_than,"(0.7022545372126158, 0.70281254107079)","(0.9061745373054381, 0.9070556489907412)","(0.6785905538599081, 0.6926440530235777)","(0.8302443534492582, 0.8326871122836083)","(0.5218689488593867, 0.5235643844739467)","(0.7770892378514803, 0.7785774288151864)","(0.6404244885173523, 0.6441908960980322)"
5,true_false-common_claim_true_false,"(0.652731344058006, 0.6554896694203769)","(0.9086079717558211, 0.9109689148876903)","(0.5363432760476853, 0.5470795215525673)","(0.6435162681020676, 0.6638874337488576)","(0.5039865318926914, 0.5040801347739753)","(0.8019012276862538, 0.8087654389804124)","(0.5393510396585797, 0.5506155155253665)"
6,true_false-counterfact_true_false,"(0.771873103978727, 0.7729339385252745)","(0.939225991667795, 0.9395867648841467)","(0.531097004004342, 0.533727751284575)","(0.9084193605392191, 0.9091644313567285)","(0.5048574633051688, 0.5050758700281648)","(0.8363343245198588, 0.8396656754801409)","(0.49707457628900553, 0.502256527389924)"
7,cross_topic-multi_topic_affirm_neg,"(0.738609443982534, 0.7442966371762197)","(0.9771133540268268, 0.9780929328101083)","(0.9482773459711572, 0.9524593927321826)","(0.709254930271086, 0.7194844394137565)","(0.6556337477191285, 0.6653329189475381)","(0.8294971844669943, 0.8391694821996725)","(0.7818411310885977, 0.7944130495134093)"
8,cross_topic-multi_topic_affirm_neg_conj,"(0.7229312726166237, 0.7324086098152223)","(0.9752012061827592, 0.9780404441119362)","(0.9593528355294778, 0.9647630780265144)","(0.8975254941626619, 0.9172863553169218)","(0.5255350588876276, 0.5450649411123726)","(0.8281368389377525, 0.8348631610622473)","(0.8000284051233619, 0.8093361433716213)"
9,cross_topic-multi_topic_affirm_neg_plus_smalle...,"(0.7269054398560973, 0.730591719483121)","(0.979748252249979, 0.9804678577696675)","(0.9741872505126128, 0.9775809223754028)","(0.7400152540032336, 0.7510052562518941)","(0.5369021502232824, 0.5612978497767175)","(0.7941401298915055, 0.8011932034418278)","(0.7996366856260714, 0.804711140460885)"


It's a little surprising that adding "smaller than" and "common claim true false" to the multi-topic-affirm-neg scenario actually resulted in a small but statistically significant _drop_ in average generalization accuracy (across all not-seen-in-training datasets).

Likewise, it's strange that the "smaller than" scenario's probe generalized quite a bit better than the "larger than" scenario's probe (by ~9pp).

In [200]:
ttpd_acc_advantage_on_multi_topic_unseen_data_groupings_df

,scenario,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
0,cities-cities,"(-0.03209030452598348, -0.00667480121419781)","(0.003608303933459447, 0.03217864998634907)","(0.2091199648866019, 0.2676088005065146)","(-0.04015518288470083, 0.011090650618567783)","(0.00324502294502349, 0.06905497705497653)","(-0.02021103704662324, -0.0004556296200434271)","(0.11699242048456837, 0.17230523837830786)"
1,cities-cities_conj,"(-0.047430925204041144, -0.02152938296469636)","(-0.10382504712573572, -0.08180804412781657)","(-0.016524292803899852, -0.010567592260830119)","(0.027651264564917562, 0.059188735435082476)","(-0.06430949293313026, -0.027957173733536393)","(-0.05695239163269605, -0.027380941700637262)","(-0.018150686801738907, -0.005929580756789507)"
2,cities-neg_cities,"(0.07357343304322705, 0.10424189928305694)","(0.13117178009525898, 0.2353896344926792)","(-0.030738889782299203, -0.019291032419974867)","(0.16584043340264082, 0.18783640501656879)","(-0.018242251197232974, 0.0002755845305663345)","(0.06830168919453339, 0.12703164413879992)","(0.01711129883288482, 0.02971144364203155)"
3,relative_comparison-larger_than,"(-0.016569116852391624, -0.000634938334741559)","(-0.18798966189102925, -0.16164721843735722)","(0.045225722892380714, 0.05730032699710464)","(0.21674779245082626, 0.25271193741410614)","(0.002644255425829507, 0.0206557445741705)","(-0.01855013500344213, -0.002449864996557853)","(0.059296166651102704, 0.07347975308133872)"
4,relative_comparison-smaller_than,"(-0.019889440698968884, -0.012125194562865128)","(-0.02488997239116099, -0.01540052334612979)","(-0.11588910139063927, -0.05316678746316561)","(0.11161091694676803, 0.12513245473907494)","(-0.1125484069707075, -0.10208492636262587)","(-0.03223874385577269, -0.025761256144227376)","(-0.06048858013053476, -0.045531486759097274)"
5,true_false-common_claim_true_false,"(-0.09611553913796524, -0.08909223592902667)","(-0.020291353334897205, -0.01564170634303142)","(-0.2546213948120692, -0.23802148488480485)","(-0.2233984315999046, -0.18062858190684875)","(-0.016496286043500296, -0.0051370472898330626)","(-0.04684369160375274, -0.03715630839624726)","(-0.09848782816999478, -0.0770974561109418)"
6,true_false-counterfact_true_false,"(0.06036758730324806, 0.08068700988686399)","(0.02043048747206035, 0.023712234346695577)","(0.08791533603235183, 0.10109634694838707)","(0.06488410703715887, 0.09709688345808874)","(-0.12165610616799072, -0.0853772271653426)","(-0.014004091885839107, -0.00566257478082756)","(0.03482739565971177, 0.0665103969824287)"
7,cross_topic-multi_topic_affirm_neg,"(-0.009463044456490185, -0.0018658439214843145)","(0.0005990689772560171, 0.0022496540089915256)","(-0.030382847841977683, -0.024184932118729647)","(-0.029183371280783778, -0.010311376092903064)","(0.04763204055188176, 0.08786795944811825)","(-0.01911687469987368, -0.007216458633459661)","(-0.07243075081201487, -0.04964282778330277)"
8,cross_topic-multi_topic_affirm_neg_conj,"(-0.03970045162150778, -0.029996159417273925)","(-0.00017667390577003018, 0.0033692082869095303)","(-0.017730673962097217, -0.011296830949494166)","(-0.052108898731714585, -0.030437138097748985)","(0.014026063494709272, 0.0349072698386241)","(-0.028275867306298965, -0.01639079936036768)","(-0.02340852662430665, 0.0006660517079187071)"
9,cross_topic-multi_topic_affirm_neg_plus_smalle...,"(-0.04024495805265662, -0.03537780219557348)","(-0.0018468763577569685, -0.0007562670607106219)","(-0.00920414078178181, -0.006070908334131741)","(-0.11181455040864112, -0.09266268821066855)","(0.006877036196883559, 0.03132296380311641)","(-0.06664858037555885, -0.05535141962444117)","(-0.06273138375690133, -0.04763650921968722)"


The results here for TTPD probes' test-set generalization (when compared with the generalization of LR probes that were trained on the same data as them) do not show a clear advantage for TTPD over LR probes, but they do show a closer competition between the two than was seen when comparing training-set or validation-set accuracies. Here, there are many cases where TTPD probes' generalization accuracy appears slightly better than LR probes' generalization accuracy and many others where the 95% confidence interval includes 0 (i.e. it isn't clear which is better).

As with data variants, 'unseen' results for a target category aren't completely comparable between scenarios if one scenario was trained on some data from that category and the other wasn't.

In [201]:
ttpd_acc_on_single_topic_unseen_data_groupings_df

,scenario,animal_class,cities,element_symb,facts,inventors,real_world_scenarios,relative_comparison,sp_en_trans,true_false
0,cities-cities,"(0.7337986346155444, 0.7450529060007024)","(0.7793914326251209, 0.8134299363231262)","(0.6670839443328598, 0.6716388817540966)","(0.6476575158932778, 0.653873326069802)","(0.577967367396217, 0.5831204987125699)","(0.7619385517354896, 0.8148826403174908)","(0.7127845649882544, 0.7753467481430588)","(0.6146321442600154, 0.6350913070674183)","(0.5940870850889864, 0.609870184093196)"
1,cities-cities_conj,"(0.7187542440570955, 0.7201813301725961)","(0.6443049258401164, 0.6584511988814873)","(0.7222433808033447, 0.7241560757183946)","(0.6722110622344566, 0.6753350881482538)","(0.5895873073586031, 0.5953499311351207)","(0.7268508758839132, 0.7658643559041662)","(0.9044352000974689, 0.9277365170742482)","(0.740053285705058, 0.7426900771268005)","(0.6373053775281983, 0.6497106053355355)"
2,cities-neg_cities,"(0.7053166006158064, 0.7142212145102439)","(0.8147654214976106, 0.815841145002111)","(0.7382021686677176, 0.7412135922018475)","(0.6734033272335433, 0.679590819547186)","(0.642901483425899, 0.6493056295448121)","(0.7932793652512459, 0.800098118192463)","(0.6860929451144718, 0.6979474589259321)","(0.8250138280907549, 0.8260923665995106)","(0.6646209385078605, 0.6664055897505019)"
3,relative_comparison-larger_than,"(0.7171338680720154, 0.7187905016758838)","(0.6486319129733807, 0.6641971636474441)","(0.6378234798783482, 0.6575569549042604)","(0.6093388408151118, 0.6201974041196022)","(0.5507072332586144, 0.5587592939380385)","(0.5784496969329992, 0.6116165282325637)","(0.8367322661094304, 0.8550859157087515)","(0.6799356935053533, 0.6934603241937618)","(0.5731358188842343, 0.5819528832632913)"
4,relative_comparison-smaller_than,"(0.7311821962453893, 0.7362547785445268)","(0.8621481416480522, 0.8709756304541093)","(0.6686142899788646, 0.6694971230646135)","(0.6846783737354939, 0.6861005546751319)","(0.6099182786223409, 0.6115775372772405)","(0.8366521130026086, 0.8414935823616294)","(0.8706360187786738, 0.874060950918296)","(0.6771482650690919, 0.680097310152147)","(0.6763164934234333, 0.677659999134374)"
5,true_false-common_claim_true_false,"(0.5306767733830728, 0.5356397532275716)","(0.670975330297474, 0.6820305636121528)","(0.698107107868801, 0.7039988703920685)","(0.6515635129418396, 0.6545148301468593)","(0.6217338398846438, 0.6316657417053143)","(0.6718699820199908, 0.6890571702978899)","(0.5001030311614082, 0.5004020193436424)","(0.6802450900428896, 0.6933168568597652)","(0.6719120390706129, 0.6732669122496225)"
6,true_false-counterfact_true_false,"(0.7302242628548999, 0.7313303589938392)","(0.7946786274120563, 0.796323337224486)","(0.7071713426626666, 0.7079101790764638)","(0.6823681943879364, 0.6833679604972503)","(0.6438744726723266, 0.6445146486665857)","(0.8536158758266852, 0.8563178990077522)","(0.9496145273576883, 0.9535420382988778)","(0.7338683789542404, 0.7349369307802729)","(0.7326741714844137, 0.7348539184032267)"
7,cross_topic-multi_topic_affirm_neg,"(0.6696925686162144, 0.6994892495656037)","(0.8728898747528887, 0.875066903243182)","(0.774562954886919, 0.7852875885913421)","(0.6516802384656153, 0.6608766314160952)","(0.639129530438257, 0.6576886513799244)","(0.8051744158157428, 0.8246269086875684)","(0.7757755315380502, 0.7989466906841717)","(0.8071811316701927, 0.8130069214271525)","(0.7170293163885477, 0.7225049286820295)"
8,cross_topic-multi_topic_affirm_neg_conj,"(0.5684505850212354, 0.5705494149787647)","(0.912750893269847, 0.9204711106594262)","(0.7471872396796602, 0.7549866733638182)","(0.5982202930120122, 0.613131960744248)","(0.5722136970381391, 0.624286302961861)","(0.8381333720232866, 0.850608349831018)","(0.7972625219299341, 0.8169041447367325)","(0.8133243713081276, 0.8185340357715182)","(0.6873702367693013, 0.6997418629725562)"
9,cross_topic-multi_topic_affirm_neg_plus_smalle...,"(0.6539948958094124, 0.6676414678269513)","(0.8740090030113684, 0.8773073034719322)"

In [202]:
ttpd_acc_advantage_on_single_topic_unseen_data_groupings_df

,scenario,animal_class,cities,element_symb,facts,inventors,real_world_scenarios,relative_comparison,sp_en_trans,true_false
0,cities-cities,"(0.007964698170086722, 0.08132101611562759)","(0.14236170076662688, 0.21278576723559908)","(-0.01627188837477974, 0.03257623620086668)","(0.05307207699847936, 0.07079104772011588)","(0.009384211692802989, 0.04129570462518867)","(0.04482312705845195, 0.11610402525942884)","(0.0647487164830864, 0.16686744513307528)","(-0.04770890828519257, 0.020441209170148304)","(-0.07759561120213268, -0.05271415976507768)"
1,cities-cities_conj,"(-0.05849208408334533, -0.03891687950208886)","(-0.0716577141162954, -0.0552910609393839)","(-0.010820695175477263, 0.0074918908276511535)","(-0.009728563085248897, 0.00891811734008906)","(-0.056435877774165664, -0.03833399670282176)","(-0.12083360773562826, -0.07784188895311353)","(0.002384204037122882, 0.0982218565689377)","(-0.0014051627973262846, 0.010863127399096215)","(-0.05812049499058826, -0.03125447068195529)"
2,cities-neg_cities,"(0.09842401817224214, 0.12601575773812201)","(0.19091293015296995, 0.2539117776934408)","(0.04094346369285068, 0.08779294935062754)","(0.04619200539063471, 0.07424923729284118)","(-0.023988586560954164, -0.004097187497623242)","(0.0793676161482437, 0.17824827789149136)","(0.13160512094949628, 0.17735952551515022)","(0.04044613338602151, 0.06325961882636788)","(0.05949241193430027, 0.09552214290724415)"
3,relative_comparison-larger_than,"(0.055483285980710076, 0.09129542550388374)","(-0.012814057111293548, 0.007096964773376063)","(0.07659308884671223, 0.10146397637067905)","(-0.01323521878998233, 0.0076971728647234365)","(0.050869404688083056, 0.06612850326170772)","(0.03909676219154972, 0.07878403251043703)","(0.18202137302946714, 0.2179281219200278)","(0.1368985244903365, 0.16376519232382283)","(-0.04814673267161387, -0.031770332193547884)"
4,relative_comparison-smaller_than,"(0.0011245651572831996, 0.01358131719565797)","(0.04098742688237714, 0.07820707213530459)","(-0.08744774443063248, -0.07274247296067189)","(-0.024922582754248426, -0.01857133890266289)","(-0.06593161444304169, -0.060062109406330666)","(0.039727070712288495, 0.06821994915526121)","(0.138566139845188, 0.16507022379117559)","(-0.13428082466844488, -0.12230103373863482)","(-0.028609211162726488, -0.023238979093768264)"
5,true_false-common_claim_true_false,"(-0.23494824836317016, -0.18360917460601753)","(-0.252904886292888, -0.23029747127096262)","(-0.016281222215205143, -0.0016535603934905354)","(-0.0612912322606771, -0.05316171686134002)","(0.013399528041476672, 0.036025158150991936)","(-0.10233875287492006, -0.026138068317132925)","(-0.45665168629633307, -0.39569679855215173)","(-0.06584990873399252, -0.03846425055804291)","(-0.042555120342752564, -0.03362120302103168)"
6,true_false-counterfact_true_false,"(-0.029986343848575136, -0.01665231161360981)","(0.016387917718783063, 0.02846473454054895)","(0.035969454964910746, 0.055878371122045745)","(0.07560116360583025, 0.08711833211681719)","(-0.04215212000372016, -0.028402273301719135)","(0.2232265365827772, 0.25359465547020305)","(0.1409830059517202, 0.23391598394726965)","(0.06770943687674622, 0.07935914719405024)","(0.08824584920335128, 0.0975294316955251)"
7,cross_topic-multi_topic_affirm_neg,"(0.0012245117195815655, 0.04023003373496389)","(0.0012920416729508177, 0.007705993690506962)","(0.01569934981865987, 0.0366783675726445)","(-0.01815250006218303, -0.002502640975123635)","(0.004862369096858173, 0.042319449084960006)","(-0.02287483280930238, 0.012278806319236125)","(-0.1163571052921821, -0.06679946036438358)","(-0.03225042151615647, -0.022395596182958576)","(-0.0024916797917875315, 0.003499533913828506)"
8,cross_topic-multi_topic_affirm_neg_conj,"(0.005070253975000907, 0.007429746024999094)","(-0.014522744249604204, -0.006832854964541146)","(-0.009192048587153695, 0.0013795485871537184)","(0.0023450468710431507, 0.022529744447821624)","(0.02587694751963679, 0.08245638581369656)","(0.07673609173624704, 0.

In [203]:
(overall_test_set_accuracy_advantage_for_ttpd:= calc_conf_interval(safely_concat_ndarrays(all_scenarios_all_unseen_datasets_acc_advantages)))

(-0.01438359,-0.007158689)

This overall test-set generalization deficit (of TTPD relative to LR) of ~1pp seems markedly more congruent with Burger et al's finding (of a 0.7pp difference in means that wasn't statistically significant because the 95% confidence intervals overlapped) than earlier comparisons of train-set or validation-set performance.